# Vilier PhoWhisper ASR on Kaggle GPU

Notebook này được viết để mở ở local nhưng chạy bằng Kaggle GPU kernel. Vì cell chạy trên Kaggle nên notebook không nhìn thấy filesystem local; kết quả được đóng gói thành zip trong `/kaggle/working` và hiển thị link download.

Input cần add vào Kaggle Notebook dưới dạng Dataset chứa một file zip bundle từ local `outputs/<audio_id>/`, gồm `manifest.timeline.json`, `vad.json`, và `asr_audio/*.wav`. Bundle cũ chưa có `asr_segments` vẫn có thể fallback sang `vad_audio/*.wav`.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import zipfile
from IPython.display import FileLink, HTML, display

# Required: set these before running on Kaggle.
REPO_URL = "https://github.com/ngocbao220/vilier.git"
REPO_REF = "main"
AUDIO_ID = "haveasip_khanhvi_5m"

# Optional: leave empty to auto-pick the first zip under /kaggle/input.
BUNDLE_ZIP_NAME = ""

# Kaggle GPU device. Use "cpu" only for debugging without GPU.
ASR_DEVICE = "0"

KAGGLE_INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")
REPO_DIR = WORK_ROOT / "vilier"
BUNDLE_DIR = WORK_ROOT / "asr_bundle" / AUDIO_ID
ASR_OUTPUT_DIR = WORK_ROOT / "asr_result"
RESULT_ZIP = WORK_ROOT / f"{AUDIO_ID}_asr_result.zip"

print("audio_id:", AUDIO_ID)
print("repo:", REPO_URL)
print("bundle_zip_name:", BUNDLE_ZIP_NAME or "<auto>")


audio_id: podcast_single_30s
repo: https://github.com/YOUR_USER/YOUR_VILIER_REPO.git
bundle_zip_name: <auto>


## 1. Check GPU

In [2]:
try:
    print(subprocess.check_output(["nvidia-smi"], text=True))
except Exception as exc:
    raise RuntimeError("Kaggle GPU is not visible. Enable GPU accelerator before running ASR.") from exc


Tue Aug 25 02:42:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Clone repo and install ASR dependencies

In [3]:
if not REPO_URL or "YOUR_USER" in REPO_URL:
    raise ValueError("Set REPO_URL to the Vilier Git repository before running this notebook.")

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_REF], check=True)
sys.path.insert(0, str(REPO_DIR))

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "transformers", "accelerate", "soundfile", "librosa", "pyyaml"],
    check=True,
)
print("repo cloned to", REPO_DIR)


ValueError: Set REPO_URL to the Vilier Git repository before running this notebook.

## 3. Unpack ASR input bundle

In [ ]:
def find_bundle_zip() -> Path:
    if BUNDLE_ZIP_NAME:
        matches = sorted(KAGGLE_INPUT_ROOT.rglob(BUNDLE_ZIP_NAME))
        if not matches:
            raise FileNotFoundError(f"Could not find {BUNDLE_ZIP_NAME} under {KAGGLE_INPUT_ROOT}")
        return matches[0]
    matches = sorted(KAGGLE_INPUT_ROOT.rglob(f"{AUDIO_ID}*_asr_bundle.zip"))
    if not matches:
        matches = sorted(KAGGLE_INPUT_ROOT.rglob("*.zip"))
    if not matches:
        raise FileNotFoundError(f"No zip bundle found under {KAGGLE_INPUT_ROOT}")
    return matches[0]

bundle_zip = find_bundle_zip()
if BUNDLE_DIR.exists():
    shutil.rmtree(BUNDLE_DIR)
BUNDLE_DIR.mkdir(parents=True)
with zipfile.ZipFile(bundle_zip) as zf:
    zf.extractall(BUNDLE_DIR)

required_paths = [BUNDLE_DIR / "manifest.timeline.json", BUNDLE_DIR / "vad.json"]
missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError("Bundle is missing required files: " + ", ".join(missing))
manifest = json.loads((BUNDLE_DIR / "manifest.timeline.json").read_text(encoding="utf-8"))
asr_files = sorted((BUNDLE_DIR / "asr_audio").rglob("*.wav"))
vad_files = sorted((BUNDLE_DIR / "vad_audio").glob("*.wav"))
if manifest.get("asr_segments") and not asr_files:
    raise FileNotFoundError("Bundle has asr_segments but no asr_audio/*.wav files")
if not manifest.get("asr_segments") and not vad_files:
    raise FileNotFoundError("Bundle has no vad_audio/*.wav files")
vad_count = len(manifest.get("vad_segments", []))
asr_count = len(manifest.get("asr_segments", []))
print("bundle:", bundle_zip)
print("extracted:", BUNDLE_DIR)
print("asr wav files:", len(asr_files))
print("vad wav files:", len(vad_files))
print("manifest asr segments:", asr_count)
print("manifest vad segments:", vad_count)


## 4. Run PhoWhisper ASR on GPU

In [ ]:
if ASR_OUTPUT_DIR.exists():
    shutil.rmtree(ASR_OUTPUT_DIR)
ASR_OUTPUT_DIR.mkdir(parents=True)

cmd = [
    sys.executable,
    str(REPO_DIR / "tools" / "run_asr_bundle.py"),
    "--config",
    str(REPO_DIR / "config.json"),
    "--bundle-dir",
    str(BUNDLE_DIR),
    "--output-dir",
    str(ASR_OUTPUT_DIR),
    "--device",
    ASR_DEVICE,
]
env = os.environ.copy()
env["PYTHONPATH"] = str(REPO_DIR)
subprocess.run(cmd, check=True, env=env)

transcript_path = ASR_OUTPUT_DIR / "transcript.json"
if not transcript_path.exists():
    raise FileNotFoundError(transcript_path)
transcript = json.loads(transcript_path.read_text(encoding="utf-8"))
expected = len(manifest.get("vad_segments", []))
if expected and len(transcript) != expected:
    raise ValueError(f"Transcript count mismatch: got {len(transcript)}, expected {expected}")
required_keys = {"id", "vad_id", "audio", "start", "end", "speaker", "text", "model", "language"}
bad = [idx for idx, item in enumerate(transcript) if not required_keys.issubset(item)]
if bad:
    raise ValueError(f"Transcript records missing keys at indexes: {bad[:10]}")
print("transcripts:", len(transcript))
print("sample:", transcript[0] if transcript else None)


## 5. Package result for local download

In [ ]:
if RESULT_ZIP.exists():
    RESULT_ZIP.unlink()
with zipfile.ZipFile(RESULT_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for item in ASR_OUTPUT_DIR.rglob("*"):
        if item.is_file():
            zf.write(item, item.relative_to(ASR_OUTPUT_DIR).as_posix())

print("result zip:", RESULT_ZIP)
print("size_mb:", round(RESULT_ZIP.stat().st_size / 1024 / 1024, 3))
display(FileLink(str(RESULT_ZIP)))
display(HTML(f'<a href="{RESULT_ZIP.name}" download>Download {RESULT_ZIP.name}</a>'))
